# Bài tập 3 - Mạng Neural Đơn giản với NumPy
Trong bài thực hành này, chúng ta sẽ xây dựng một mạng neural đơn giản sử dụng NumPy. Đây chính là mạng "rang cà phê" tương tự mạng bạn đã triển khai bằng TensorFlow.
   <center> <img  src="./images/C2_W1_CoffeeRoasting.png" width="400" />   <center/>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from lab_utils_common import dlc, sigmoid
from lab_coffee_utils import load_coffee_data, plt_roast, plt_prob, plt_layer, plt_network, plt_output_unit
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

## Bộ dữ liệu (DataSet)
Đây chính là bộ dữ liệu tương tự như trong bài thực hành trước.

In [ ]:
X,Y = load_coffee_data();
print(X.shape, Y.shape)

Hãy vẽ biểu đồ dữ liệu rang cà phê bên dưới. Hai đặc trưng là Nhiệt độ theo độ C và Thời lượng theo phút. Trang [Coffee Roasting at Home](https://www.merchantsofgreencoffee.com/how-to-roast-green-coffee-in-your-oven/) gợi ý rằng thời lượng tốt nhất nên duy trì trong khoảng 12 đến 15 phút trong khi nhiệt độ nên nằm trong khoảng 175 đến 260 độ C. Tất nhiên, khi nhiệt độ tăng lên thì thời lượng nên giảm đi. 

In [ ]:
plt_roast(X,Y)

### Chuẩn hóa Dữ liệu (Normalize Data)
Để đồng nhất với bài thực hành trước, chúng ta sẽ chuẩn hóa dữ liệu. Hãy tham khảo bài thực hành đó để biết thêm chi tiết.

In [ ]:
print(f"Temperature Max, Min pre normalization: {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Duration    Max, Min pre normalization: {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")
norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)  # learns mean, variance
Xn = norm_l(X)
print(f"Temperature Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
print(f"Duration    Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")

## Mô hình NumPy (Lan truyền xuôi bằng NumPy)
<center> <img  src="./images/C2_W1_RoastingNetwork.PNG" width="200" />   <center/>  
Hãy cùng xây dựng mạng "Coffee Roasting Network". Mạng gồm có hai lớp với các hàm kích hoạt sigmoid.

Như đã mô tả trong bài giảng, bạn hoàn toàn có thể tự xây dựng lớp dense (lớp kết nối đầy đủ) bằng NumPy. Sau đó, nó có thể được sử dụng để xây dựng một mạng neural nhiều lớp. 

<img src="images/C2_W1_dense3.png" width="600" height="450">

Trong bài thực hành tùy chọn đầu tiên, bạn đã xây dựng một neuron bằng NumPy và TensorFlow đồng thời nhận thấy sự tương đồng giữa chúng. Một lớp đơn giản chỉ chứa nhiều neuron/đơn vị (units). Ta có thể sử dụng một vòng lặp `for` để duyệt qua từng đơn vị (`j`) trong lớp và thực hiện tích vô hướng (dot product) của các trọng số cho đơn vị đó (`W[:,j]`) rồi cộng với bias của đơn vị đó (`b[j]`) để tạo thành `z`. Sau đó, một hàm kích hoạt `g(z)` có thể được áp dụng cho kết quả đó. Hãy cùng thử điều đó dưới đây để xây dựng một hàm con "lớp dense" (dense layer subroutine).

Đầu tiên, bạn sẽ định nghĩa hàm kích hoạt `g()`. Bạn sẽ sử dụng hàm `sigmoid()` vốn đã được triển khai sẵn cho bạn trong tệp `lab_utils_common.py` bên ngoài notebook này.

In [ ]:
# Define the activation function
g = sigmoid

Tiếp theo, bạn sẽ định nghĩa hàm `my_dense()` dùng để tính toán các giá trị kích hoạt (activations) của một lớp dense.

In [ ]:
def my_dense(a_in, W, b):
    """
    Computes dense layer
    Args:
      a_in (ndarray (n, )) : Data, 1 example 
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j, )) : bias vector, j units  
    Returns
      a_out (ndarray (j,))  : j units|
    """
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):               
        w = W[:,j]                                    
        z = np.dot(w, a_in) + b[j]         
        a_out[j] = g(z)               
    return(a_out)

*Lưu ý: Bạn cũng có thể triển khai hàm ở trên để nhận thêm `g` làm một tham số bổ sung (ví dụ: `my_dense(a_in, W, b, g)`). Tuy nhiên, trong notebook này, bạn sẽ chỉ sử dụng một loại hàm kích hoạt duy nhất (đó là sigmoid), vì vậy việc coi nó như một hằng số và định nghĩa bên ngoài hàm là hoàn toàn phù hợp. Đó là những gì bạn đã làm trong đoạn code ở trên và nó giúp việc gọi hàm trong các ô code tiếp theo trở nên đơn giản hơn. Chỉ cần lưu ý rằng truyền nó dưới dạng một tham số cũng là một cách triển khai hợp lệ.*

Ô tiếp theo sẽ xây dựng một mạng neural hai lớp sử dụng hàm con `my_dense` ở trên.

In [ ]:
def my_sequential(x, W1, b1, W2, b2):
    a1 = my_dense(x,  W1, b1)
    a2 = my_dense(a1, W2, b2)
    return(a2)

Chúng ta có thể sao chép các trọng số và bias đã được huấn luyện từ bài thực hành TensorFlow trước đó.

In [ ]:
W1_tmp = np.array( [[-8.93,  0.29, 12.9 ], [-0.1,  -7.32, 10.81]] )
b1_tmp = np.array( [-9.82, -9.28,  0.96] )
W2_tmp = np.array( [[-31.18], [-27.59], [-32.56]] )
b2_tmp = np.array( [15.41] )

### Dự đoán (Predictions)
<img align="left" src="./images/C2_W1_RoastingDecision.PNG"     style=" width:380px; padding: 10px 20px; " >

Khi đã có một mô hình được huấn luyện, bạn có thể sử dụng nó để đưa ra dự đoán. Hãy nhớ rằng đầu ra của mô hình chúng ta là một xác suất. Trong trường hợp này, đó là xác suất mẻ rang đạt chuẩn (good roast). Để đưa ra quyết định, ta phải so sánh xác suất với một ngưỡng (threshold). Ở đây, chúng ta sẽ sử dụng ngưỡng 0.5

Hãy bắt đầu bằng cách viết một hàm tương tự như phương thức `model.predict()` của TensorFlow. Hàm này sẽ nhận một ma trận $X$ chứa toàn bộ $m$ mẫu dữ liệu theo các hàng và đưa ra dự đoán bằng cách chạy mô hình.

In [ ]:
def my_predict(X, W1, b1, W2, b2):
    m = X.shape[0]
    p = np.zeros((m,1))
    for i in range(m):
        p[i,0] = my_sequential(X[i], W1, b1, W2, b2)
    return(p)

Chúng ta có thể thử nghiệm hàm này trên hai mẫu dữ liệu:

In [ ]:
X_tst = np.array([
    [200,13.9],  # postive example
    [200,17]])   # negative example
X_tstn = norm_l(X_tst)  # remember to normalize
predictions = my_predict(X_tstn, W1_tmp, b1_tmp, W2_tmp, b2_tmp)

Để chuyển đổi các xác suất thành quyết định, chúng ta áp dụng một ngưỡng:

In [ ]:
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i] >= 0.5:
        yhat[i] = 1
    else:
        yhat[i] = 0
print(f"decisions = \n{yhat}")

Điều này có thể được thực hiện một cách ngắn gọn hơn:

In [ ]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")

## Chức năng của mạng

Biểu đồ này hiển thị hoạt động của toàn bộ mạng và cho kết quả giống hệt như kết quả từ TensorFlow trong bài thực hành trước.  
Biểu đồ bên trái là đầu ra thô của lớp cuối cùng, được biểu diễn bằng vùng màu xanh lam. Biểu đồ này được phủ lên trên dữ liệu huấn luyện (được biểu thị bằng các dấu X và O).   
Biểu đồ bên phải là đầu ra của mạng sau khi áp dụng ngưỡng quyết định. Các dấu X và O ở đây tương ứng với các quyết định do mạng đưa ra.  

In [ ]:
netf= lambda x : my_predict(norm_l(x),W1_tmp, b1_tmp, W2_tmp, b2_tmp)
plt_network(X,Y,netf)

## Chúc mừng bạn!
Bạn đã xây dựng thành công một mạng neural đơn giản bằng NumPy.

*Nguồn: Andrew Ng, Advanced Learning Algorithms, DeepLearning.AI & Standford Online.*